# Vector Similarity: by Hand, with NumPy and with Qdrant

| # | Task | Section |
|---|---|---|
| 1 | Class example by hand: points (2,3), (1,2), (0,4), query (2,4); Euclidean distance and cosine similarity; rankings | 1 |
| 2 | Brute-force top-k in NumPy for both metrics over the Gemini Embedding 2 vectors from task 4 | 2 |
| 3 | Same queries in Qdrant; show the results match NumPy | 3 |
| 4 | Normalise the vectors; show cosine and dot product give the same ranking | 4 |
| 5 | Time brute force vs Qdrant for 1,000 and 10,000 vectors; what HNSW changes | 5 |

**Setup:** `pip install numpy qdrant-client python-dotenv`, and a `.env` with `QDRANT_URL` and `QDRANT_API_KEY` (the long secret key from the cluster's **API Keys** tab).

## Import libraries

In [1]:
import math
import numpy as np
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
import time
from qdrant_client import models
from qdrant_client.models import Distance, VectorParams, PointStruct

## Settings and connection

In [3]:
load_dotenv(override=True)

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

SOURCE_COLLECTION = "sentinel_corpus"   # Gemini vectors from task 4
TEST_COLLECTION = "similarity_check"    # made in section 3
TOP_K = 5
BATCH_SIZE = 32                         # small batches = small uploads, no timeouts

if QDRANT_URL == ":memory:":
    client = QdrantClient(":memory:")   # local test mode
else:
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=300)

print("Connected. Collections:", [c.name for c in client.get_collections().collections])

Connected. Collections: ['sentinel_corpus']


In [10]:
# Upload points a few at a time, retrying if the internet is slow
def upload_in_batches(collection_name, points, batch_size=BATCH_SIZE):
    for start in range(0, len(points), batch_size):
        batch = points[start:start + batch_size]
        for attempt in range(1, 4):
            try:
                client.upsert(collection_name=collection_name, points=batch, wait=True)
                break
            except Exception as error:
                if attempt == 3:
                    raise
                print(f"  upload problem ({type(error).__name__}), retrying in {5 * attempt}s ...")
                time.sleep(5 * attempt)
        print(f"  uploaded {min(start + batch_size, len(points))}/{len(points)}", end="\r")
    print()

---
## 1. The class example by hand

* **Euclidean distance** is the straight-line gap between two points: `√((q₁−v₁)² + (q₂−v₂)²)`. **Smaller is closer.**
* **Cosine similarity** measures the angle between two arrows: `(q₁v₁ + q₂v₂) ÷ (‖q‖ × ‖v‖)`. **Bigger is closer**; 1 means the same direction.

In [11]:
# Define three stored 2D vectors
points = {
    "A": (2, 3),
    "B": (1, 2),
    "C": (0, 4),
}

# Define the query vector
query = (2, 4)

print("Stored Vectors:", points)
print("Query Vector:", query)

Stored Vectors: {'A': (2, 3), 'B': (1, 2), 'C': (0, 4)}
Query Vector: (2, 4)


In [6]:
# Calculate straight-line distance between two vectors
def euclidean_distance(v1, v2):
    return math.sqrt((v1[0] - v2[0]) ** 2 + (v1[1] - v2[1]) ** 2)


# Calculate cosine similarity between two vectors
def cosine_similarity(v1, v2):
    dot = v1[0] * v2[0] + v1[1] * v2[1]
    length1 = math.sqrt(v1[0] ** 2 + v1[1] ** 2)
    length2 = math.sqrt(v2[0] ** 2 + v2[1] ** 2)
    return dot / (length1 * length2)


euclidean_results = {}
cosine_results = {}
for name, point in points.items():
    euclidean_results[name] = euclidean_distance(query, point)
    cosine_results[name] = cosine_similarity(query, point)
    print(f"Point {name} {point}: Euclidean = {euclidean_results[name]:.4f}   Cosine = {cosine_results[name]:.4f}")

Point A (2, 3): Euclidean = 1.0000   Cosine = 0.9923
Point B (1, 2): Euclidean = 2.2361   Cosine = 1.0000
Point C (0, 4): Euclidean = 2.0000   Cosine = 0.8944


In [7]:
# Euclidean: smallest first.  Cosine: biggest first.
euclidean_rank = sorted(euclidean_results, key=euclidean_results.get)
cosine_rank = sorted(cosine_results, key=cosine_results.get, reverse=True)

print("Euclidean ranking:", " → ".join(euclidean_rank))
print("Cosine ranking:   ", " → ".join(cosine_rank))

Euclidean ranking: A → C → B
Cosine ranking:    B → A → C


**Why they differ:** B = (1, 2) is exactly half of the query (2, 4). It points the **same way**, so its cosine is a perfect 1.0, but its tip is the **furthest** away, so Euclidean ranks it last. Euclidean cares about length; cosine only cares about direction.

---
## 2. Brute-force top-k in NumPy

Brute force means comparing the query with **every** stored vector, then sorting. We read the Gemini Embedding 2 vectors (3072 numbers each) back from the task 4 collection. If that collection doesn't exist, we use 1,000 random vectors instead.

In [12]:
# Read every vector from the task 4 collection
ids, vectors, titles = [], [], []

if client.collection_exists(SOURCE_COLLECTION):
    next_page = None
    while True:
        records, next_page = client.scroll(collection_name=SOURCE_COLLECTION, limit=100,
                                           offset=next_page, with_vectors=True, with_payload=True)
        for r in records:
            ids.append(r.id)
            vectors.append(r.vector)
            titles.append(f"{r.payload.get('source_document')} #{r.payload.get('chunk_number')}")
        if next_page is None:
            break
    print("Loaded Gemini vectors from", SOURCE_COLLECTION)
else:
    rng = np.random.default_rng(42)
    vectors = rng.normal(size=(1000, 3072))
    ids = list(range(1000))
    titles = [f"random vector {i}" for i in ids]
    print("Task 4 collection not found: using 1,000 random vectors")

X = np.array(vectors, dtype=np.float32)
print("Matrix shape:", X.shape)

Loaded Gemini vectors from sentinel_corpus
Matrix shape: (425, 3072)


In [14]:
# Brute force: compare with every row, then sort
def numpy_top_k_euclidean(X, q, k=TOP_K):
    distances = np.sqrt(((X - q) ** 2).sum(axis=1))
    order = np.argsort(distances)[:k]              # smallest first
    return order, distances[order]


def numpy_top_k_cosine(X, q, k=TOP_K):
    similarities = (X @ q) / (np.linalg.norm(X, axis=1) * np.linalg.norm(q))
    order = np.argsort(-similarities)[:k]          # biggest first
    return order, similarities[order]


# 5 queries: stored vectors spread through the collection (each should find itself first)
query_rows = np.linspace(0, len(X) - 1, 5).astype(int)

numpy_results = []
for row in query_rows:
    q = X[row]
    e_order, e_scores = numpy_top_k_euclidean(X, q)
    c_order, c_scores = numpy_top_k_cosine(X, q)
    numpy_results.append({"euclid": [ids[i] for i in e_order], "euclid_scores": e_scores,
                          "cosine": [ids[i] for i in c_order], "cosine_scores": c_scores})
    print(f"\nQuery: {titles[row]}")
    print("  Euclidean top 5:", [f"{titles[i]} ({s:.4f})" for i, s in zip(e_order, e_scores)])
    print("  Cosine top 5:   ", [f"{titles[i]} ({s:.4f})" for i, s in zip(c_order, c_scores)])


Query: DOC_playbooks_0000040 #1
  Euclidean top 5: ['DOC_playbooks_0000040 #1 (0.0000)', 'DOC_playbooks_0000020 #1 (0.1749)', 'DOC_playbooks_0000041 #1 (0.2868)', 'DOC_playbooks_0000001 #1 (0.3055)', 'DOC_playbooks_0000015 #1 (0.3069)']
  Cosine top 5:    ['DOC_playbooks_0000040 #1 (1.0000)', 'DOC_playbooks_0000020 #1 (0.9847)', 'DOC_playbooks_0000041 #1 (0.9589)', 'DOC_playbooks_0000001 #1 (0.9533)', 'DOC_playbooks_0000015 #1 (0.9529)']

Query: DOC_regulatory_exercises_0000025 #1
  Euclidean top 5: ['DOC_regulatory_exercises_0000025 #1 (0.0000)', 'DOC_regulatory_exercises_0000005 #1 (0.1516)', 'DOC_regulatory_exercises_0000001 #1 (0.2657)', 'DOC_regulatory_exercises_0000021 #1 (0.2666)', 'DOC_regulatory_exercises_0000027 #1 (0.2749)']
  Cosine top 5:    ['DOC_regulatory_exercises_0000025 #1 (1.0000)', 'DOC_regulatory_exercises_0000005 #1 (0.9885)', 'DOC_regulatory_exercises_0000001 #1 (0.9647)', 'DOC_regulatory_exercises_0000021 #1 (0.9645)', 'DOC_regulatory_exercises_0000027 #1 (0.9

---
## 3. The same queries in Qdrant

One test collection stores each vector twice, under two names: `cosine` (Cosine distance) and `euclid` (Euclidean distance). We send the same 5 queries and compare with NumPy. `exact=True` makes Qdrant check every vector, like our brute force.

Points are uploaded **32 at a time with retries**, so a slow connection doesn't time out.

In [15]:
DIM = X.shape[1]

if client.collection_exists(TEST_COLLECTION):
    client.delete_collection(TEST_COLLECTION)

client.create_collection(
    collection_name=TEST_COLLECTION,
    vectors_config={
        "cosine": VectorParams(size=DIM, distance=Distance.COSINE),
        "euclid": VectorParams(size=DIM, distance=Distance.EUCLID),
    },
)

test_points = [PointStruct(id=pid, vector={"cosine": v.tolist(), "euclid": v.tolist()})
               for pid, v in zip(ids, X)]
upload_in_batches(TEST_COLLECTION, test_points)
print("Points in Qdrant:", client.count(TEST_COLLECTION, exact=True).count)

  upload problem (ResponseHandlingException), retrying in 5s ...
  upload problem (ResponseHandlingException), retrying in 5s ...
  uploaded 425/425
Points in Qdrant: 425


In [16]:
def qdrant_top_k(q, vector_name, k=TOP_K):
    hits = client.query_points(collection_name=TEST_COLLECTION, query=q.tolist(), using=vector_name,
                               limit=k, search_params=models.SearchParams(exact=True)).points
    return [h.id for h in hits], np.array([h.score for h in hits])


all_match = True
for n, row in enumerate(query_rows):
    for metric in ["euclid", "cosine"]:
        qdrant_ids, qdrant_scores = qdrant_top_k(X[row], metric)
        same_ids = qdrant_ids == numpy_results[n][metric]
        same_scores = np.allclose(qdrant_scores, numpy_results[n][f"{metric}_scores"], atol=1e-3)
        all_match = all_match and same_ids and same_scores
        print(f"Query {n + 1}, {metric:6s}: same ids and order = {same_ids}, same scores = {same_scores}")

print("\nQdrant matches NumPy for every query and both metrics:", all_match)

Query 1, euclid: same ids and order = True, same scores = True
Query 1, cosine: same ids and order = True, same scores = True
Query 2, euclid: same ids and order = True, same scores = True
Query 2, cosine: same ids and order = True, same scores = True
Query 3, euclid: same ids and order = True, same scores = True
Query 3, cosine: same ids and order = True, same scores = True
Query 4, euclid: same ids and order = True, same scores = True
Query 4, cosine: same ids and order = True, same scores = True
Query 5, euclid: same ids and order = True, same scores = True
Query 5, cosine: same ids and order = True, same scores = True

Qdrant matches NumPy for every query and both metrics: True


---
## 4. Normalise: cosine equals dot product

Cosine is the dot product divided by both lengths. If every vector is first made **length 1** (divide it by its length), dividing by 1 × 1 changes nothing, so **dot product = cosine** and the rankings are identical.

In [17]:
X2 = np.array(list(points.values()), dtype=float)
q2 = np.array(query, dtype=float)
names = list(points)

raw_dot = X2 @ q2
X2_unit = X2 / np.linalg.norm(X2, axis=1, keepdims=True)
q2_unit = q2 / np.linalg.norm(q2)
unit_dot = X2_unit @ q2_unit

for name, rd, cs, ud in zip(names, raw_dot, cosine_results.values(), unit_dot):
    print(f"{name}: raw dot = {rd:5.1f}   cosine = {cs:.4f}   dot after normalising = {ud:.4f}")

print("\nRaw dot ranking:          ", " → ".join(np.array(names)[np.argsort(-raw_dot, kind='stable')]), " (length leaks in)")
print("Dot after normalising:    ", " → ".join(np.array(names)[np.argsort(-unit_dot)]))
print("Cosine ranking:           ", " → ".join(cosine_rank))

A: raw dot =  16.0   cosine = 0.9923   dot after normalising = 0.9923
B: raw dot =  10.0   cosine = 1.0000   dot after normalising = 1.0000
C: raw dot =  16.0   cosine = 0.8944   dot after normalising = 0.8944

Raw dot ranking:           A → C → B  (length leaks in)
Dot after normalising:     B → A → C
Cosine ranking:            B → A → C


In [18]:
# The same check on all the Gemini vectors
X_unit = X / np.linalg.norm(X, axis=1, keepdims=True)
print("Lengths before:", np.linalg.norm(X, axis=1).min().round(4), "to", np.linalg.norm(X, axis=1).max().round(4))

same = True
for row in query_rows:
    cosine_order, _ = numpy_top_k_cosine(X, X[row])
    dot_order = np.argsort(-(X_unit @ X_unit[row]))[:TOP_K]
    same = same and list(cosine_order) == list(dot_order)
print("Cosine ranking == dot-product ranking after normalising, for every query:", same)

Lengths before: 1.0 to 1.0
Cosine ranking == dot-product ranking after normalising, for every query: True


---
## 5. Timing: brute force vs Qdrant (1,000 and 10,000 vectors)

We create random length-1 vectors of **768 numbers** for the timing test. 768 is one of Gemini Embedding 2's supported sizes, and it keeps the 10,000-vector upload a quarter of the size of 3072-number vectors, so it works on a normal home connection. The comparison is the same: how search time grows as the number of vectors grows 10×.

For each size, we average 20 queries:
* **NumPy brute force:** every vector, in local memory
* **Qdrant exact:** every vector, on the server
* **Qdrant HNSW:** Qdrant's normal search, using its graph index
* **recall@5:** how many of HNSW's top 5 are also in the exact top 5

In [19]:
TIMING_DIM = 768
NUM_QUERIES = 20
rng = np.random.default_rng(7)
results = []

for n in [1_000, 10_000]:
    name = f"timing_{n}"
    data = rng.normal(size=(n, TIMING_DIM)).astype(np.float32)
    data = data / np.linalg.norm(data, axis=1, keepdims=True)
    test_queries = rng.normal(size=(NUM_QUERIES, TIMING_DIM)).astype(np.float32)
    test_queries = test_queries / np.linalg.norm(test_queries, axis=1, keepdims=True)

    # Upload
    if client.collection_exists(name):
        client.delete_collection(name)
    client.create_collection(collection_name=name, vectors_config=VectorParams(size=TIMING_DIM, distance=Distance.COSINE))
    print(f"Uploading {n:,} vectors ...")
    upload_in_batches(name, [PointStruct(id=i, vector=v.tolist()) for i, v in enumerate(data)], batch_size=100)

    # Wait until Qdrant has finished building its index
    while client.get_collection(name).status != models.CollectionStatus.GREEN:
        time.sleep(2)

    # NumPy brute force
    start = time.perf_counter()
    exact_answers = [set(np.argsort(-(data @ q))[:TOP_K].tolist()) for q in test_queries]
    numpy_ms = (time.perf_counter() - start) / NUM_QUERIES * 1000

    # Qdrant exact
    start = time.perf_counter()
    for q in test_queries:
        client.query_points(collection_name=name, query=q.tolist(), limit=TOP_K,
                            search_params=models.SearchParams(exact=True))
    exact_ms = (time.perf_counter() - start) / NUM_QUERIES * 1000

    # Qdrant HNSW (the default)
    start = time.perf_counter()
    hnsw_answers = []
    for q in test_queries:
        hits = client.query_points(collection_name=name, query=q.tolist(), limit=TOP_K).points
        hnsw_answers.append({h.id for h in hits})
    hnsw_ms = (time.perf_counter() - start) / NUM_QUERIES * 1000

    recall = np.mean([len(a & b) / TOP_K for a, b in zip(hnsw_answers, exact_answers)])
    results.append((n, numpy_ms, exact_ms, hnsw_ms, recall))

print(f"\n{'Vectors':>8} | {'NumPy brute force':>17} | {'Qdrant exact':>12} | {'Qdrant HNSW':>11} | {'recall@5':>8}")
for n, a, b, c, r in results:
    print(f"{n:>8,} | {a:>14.2f} ms | {b:>9.2f} ms | {c:>8.2f} ms | {r:>8.2f}")

Uploading 1,000 vectors ...
  uploaded 1000/1000
Uploading 10,000 vectors ...
  upload problem (ResponseHandlingException), retrying in 5s ...
  upload problem (ResponseHandlingException), retrying in 5s ...
  upload problem (ResponseHandlingException), retrying in 10s ...
  upload problem (ResponseHandlingException), retrying in 5s ...
  uploaded 10000/10000

 Vectors | NumPy brute force | Qdrant exact | Qdrant HNSW | recall@5
   1,000 |           0.43 ms |    159.98 ms |   158.49 ms |     1.00
  10,000 |          14.13 ms |    173.91 ms |   170.78 ms |     0.74


### What an HNSW index changes

* **Brute force** compares the query with **every** vector. 10× more vectors means about 10× more work. It is always exact.
* **HNSW** (Hierarchical Navigable Small World) is a graph that Qdrant builds over the vectors. Each vector is linked to a few close neighbours, and the links are stacked in layers, from long jumps at the top to short steps at the bottom.
* A search starts at the top and **jumps towards closer and closer points**, so it looks at only a small part of the vectors. Its time grows roughly with the **logarithm** of the number of vectors, not in a straight line.
* It is **approximate**: it can sometimes miss a true neighbour, but recall is usually above 0.95 and can be raised with settings (`ef`, `m`). `exact=True` turns it off.
* Building the graph takes time and memory, which is why a new collection briefly shows status "yellow" before searches use the index.

**Reading our table:** NumPy is very fast at these sizes because everything is in local memory, and every Qdrant Cloud query also pays a network round trip. The important part is how each method grows from 1,000 to 10,000. At millions of vectors, only HNSW keeps search fast.

*Qdrant's local `:memory:` mode has no HNSW, so run this section against Qdrant Cloud.*

In [21]:
# Remove the test collections made by this notebook (task 4's collection is kept)
for name in [TEST_COLLECTION, "timing_1000", "timing_10000"]:
    if client.collection_exists(name):
        client.delete_collection(name)
print("Remaining collections:", [c.name for c in client.get_collections().collections])

Remaining collections: ['sentinel_corpus']
